# Embedding làm được gì ngoài search: phân loại, trùng lặp, bất thường, gợi ý

Cùng một embedding `bge-m3`, bốn việc với bốn con số: phân loại cảm xúc UIT-VSFC bằng embedding + một lớp tuyến tính
(so với TF-IDF, BiLSTM của module 04 và prompting `qwen2.5:3b` của module 11), tìm câu trùng gần đúng và rò rỉ giữa
train và test, phát hiện câu "lạ" bằng khoảng cách kNN, và gợi ý điều luật liên quan trên Zalo Legal.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell `explore` cần Ollama với `bge-m3` (embed khoảng 16 000 câu UIT-VSFC, vài phút, cache ở
  `data/raw/uit-vsfc/cache/`) và cache Zalo của bài Intermediate 01 cho phần gợi ý.
- Chấm: `uv run pytest tracks/ai-engineer/modules/12-embeddings-vector-db/exercises/intermediate/03-embedding-use-cases -q`

In [ ]:
# explore: tìm gốc repo, tải UIT-VSFC và Zalo Legal nếu chưa có
import os
import sys
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "4")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

VSFC = ensure("uit-vsfc")
ZALO = ensure("zalo-legal")
(VSFC / "cache").mkdir(exist_ok=True)
print(VSFC, ZALO)

In [ ]:
import numpy as np

## 1. Phân loại bằng embedding

Mọi vector trong bài đã chuẩn hoá L2, nên tích vô hướng là cosine. Hai bộ phân loại không cần train gì ngoài lưu vector:

- `class_centroids(x, y, n_classes) -> np.ndarray`: với mỗi lớp `c`, trung bình các vector có nhãn `c` rồi **chuẩn hoá
  L2** (lớp không có mẫu: vector 0). Trả `(n_classes, d)` float32. `predict_centroid(centroids, x)`: nhãn có cosine lớn
  nhất (bằng nhau thì nhãn nhỏ).
- `knn_predict(train_x, train_y, x, k, n_classes, batch=1024) -> np.ndarray`: với mỗi dòng của `x`, lấy `k` vector train
  có cosine lớn nhất (bằng nhau thì chỉ số nhỏ), bỏ phiếu theo nhãn. Hoà phiếu thì lớp có **tổng cosine** của các phiếu
  lớn hơn thắng; vẫn hoà thì nhãn nhỏ. Tính theo lô `batch` dòng của `x` để ma trận cosine không vượt
  `batch × len(train_x)`.

Bộ phân loại thứ ba là logistic regression của scikit-learn trên embedding (cell `explore`), thứ AI Engineer hay gọi
là "linear probe" hay "embedding + linear head".

In [ ]:
def class_centroids(x: np.ndarray, y: np.ndarray, n_classes: int) -> np.ndarray:
    """Trung bình vector mỗi lớp, chuẩn hoá L2; lớp rỗng là vector 0."""
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y)
    out = np.zeros((n_classes, x.shape[1]), dtype=np.float32)
    for c in range(n_classes):
        if (y == c).any():
            m = x[y == c].mean(axis=0)
            n = np.linalg.norm(m)
            out[c] = m / n if n > 0 else m
    return out


def predict_centroid(centroids: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Nhãn của centroid có cosine lớn nhất."""
    return np.argmax(np.asarray(x, dtype=np.float32) @ centroids.T, axis=1)


def knn_predict(train_x: np.ndarray, train_y: np.ndarray, x: np.ndarray, k: int, n_classes: int,
                batch: int = 1024) -> np.ndarray:
    """Bỏ phiếu k láng giềng cosine; hoà thì tổng cosine lớn hơn, rồi nhãn nhỏ hơn."""
    train_y = np.asarray(train_y)
    out = np.empty(len(x), dtype=np.int64)
    for s in range(0, len(x), batch):
        sims = np.asarray(x[s:s + batch], dtype=np.float32) @ np.asarray(train_x, dtype=np.float32).T
        top = np.argsort(-sims, axis=1, kind="stable")[:, :k]
        for r, idx in enumerate(top):
            votes = np.bincount(train_y[idx], minlength=n_classes)
            mass = np.bincount(train_y[idx], weights=sims[r, idx], minlength=n_classes)
            out[s + r] = max(range(n_classes), key=lambda c: (votes[c], mass[c], -c))
    return out

In [ ]:
# explore: embed UIT-VSFC (cache theo split, kèm kiểm số câu)
import json
import time

import llmkit
import pandas as pd

splits = {s: pd.read_parquet(VSFC / "default" / s / "0000.parquet") for s in ("train", "validation", "test")}


def embed_split(name: str) -> np.ndarray:
    path = VSFC / "cache" / f"bge-m3-{name}.npy"
    texts = splits[name]["sentence"].tolist()
    if path.exists():
        m = np.load(path)
        if len(m) == len(texts):
            return m
        raise RuntimeError(f"{path} không khớp số câu của split {name}: xoá tệp này rồi chạy lại")
    t0 = time.perf_counter()
    m = np.concatenate([llmkit.embed(texts[i:i + 64]) for i in range(0, len(texts), 64)]).astype(np.float32)
    m /= np.linalg.norm(m, axis=1, keepdims=True)
    print(f"embed {name}: {len(texts)} câu trong {time.perf_counter() - t0:.0f} s")
    np.save(path, m)
    return m


emb = {s: embed_split(s) for s in splits}
y = {s: splits[s]["sentiment"].to_numpy() for s in splits}
for s in splits:
    print(s, emb[s].shape, np.bincount(y[s]))

In [ ]:
# explore: ba bộ phân loại trên toàn bộ test và trên 150 câu ghim của module 11
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

test = splits["test"]
pinned = pd.concat([test[test.sentiment == c].sample(50, random_state=0) for c in (0, 1, 2)])
pin_idx = pinned.index.to_numpy()

best = None
for C in (0.1, 1.0, 10.0):
    clf = LogisticRegression(C=C, max_iter=3000)
    clf.fit(emb["train"], y["train"])
    f1 = f1_score(y["validation"], clf.predict(emb["validation"]), average="macro")
    print(f"LogReg C={C:5}: macro-F1 validation {f1:.3f}")
    if best is None or f1 > best[0]:
        best = (f1, C)
t0 = time.perf_counter()
clf = LogisticRegression(C=best[1], max_iter=3000).fit(emb["train"], y["train"])
fit_s = time.perf_counter() - t0
t0 = time.perf_counter()
pred_lr = clf.predict(emb["test"])
pred_ms = 1000 * (time.perf_counter() - t0) / len(test)

pred_knn = knn_predict(emb["train"], y["train"], emb["test"], 10, 3)
pred_cen = predict_centroid(class_centroids(emb["train"], y["train"], 3), emb["test"])

rows = []
for name, pred in [(f"embedding + LogReg (C={best[1]})", pred_lr), ("kNN k=10", pred_knn), ("centroid", pred_cen)]:
    rows.append({"bộ phân loại": name,
                 "macro-F1 test (3 166)": f1_score(y["test"], pred, average="macro"),
                 "accuracy test": accuracy_score(y["test"], pred),
                 "F1 neutral": f1_score(y["test"], pred, average=None)[1],
                 "macro-F1 150 ghim": f1_score(y["test"][pin_idx], pred[pin_idx], average="macro")})
display(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"LogReg: fit {fit_s:.1f} s trên {len(y['train'])} câu; dự đoán {pred_ms:.3f} ms/câu (chưa tính embed câu)")
print("mốc: module 04 TF-IDF + LogReg 0.716, BiLSTM 0.755 (test đầy đủ); module 11 qwen2.5:3b zero-shot 0.804 (150 ghim)")

## 2. Trùng lặp gần đúng và rò rỉ train/test

- `near_duplicates(x, threshold, batch=2048) -> list[tuple[int, int, float]]`: mọi cặp `i < j` có cosine `>= threshold`,
  kèm cosine (kiểu `float`), sắp theo cosine giảm dần rồi `(i, j)` tăng dần. Tính theo lô dòng, không dựng ma trận
  `n × n` một lần.
- `nearest_in(ref, x, batch=2048) -> tuple[np.ndarray, np.ndarray]`: với mỗi dòng của `x`, chỉ số dòng gần nhất trong
  `ref` (bằng nhau thì chỉ số nhỏ) và cosine tương ứng.

In [ ]:
def near_duplicates(x: np.ndarray, threshold: float, batch: int = 2048) -> list[tuple[int, int, float]]:
    """Mọi cặp i < j có cosine >= threshold, giảm dần theo cosine rồi (i, j) tăng."""
    x = np.asarray(x, dtype=np.float32)
    pairs = []
    for s in range(0, len(x), batch):
        sims = x[s:s + batch] @ x.T
        rows, cols = np.nonzero(sims >= threshold)
        for r, c in zip(rows, cols):
            i = s + int(r)
            if i < c:
                pairs.append((i, int(c), float(sims[r, c])))
    return sorted(pairs, key=lambda p: (-p[2], p[0], p[1]))


def nearest_in(ref: np.ndarray, x: np.ndarray, batch: int = 2048) -> tuple[np.ndarray, np.ndarray]:
    """Chỉ số dòng gần nhất trong ref cho mỗi dòng của x, và cosine của nó."""
    ref = np.asarray(ref, dtype=np.float32)
    idx, sim = [], []
    for s in range(0, len(x), batch):
        sims = np.asarray(x[s:s + batch], dtype=np.float32) @ ref.T
        j = np.argmax(sims, axis=1)
        idx.append(j)
        sim.append(sims[np.arange(len(j)), j])
    return np.concatenate(idx).astype(np.int64), np.concatenate(sim)

In [ ]:
# explore: trùng lặp trong train, và câu test có "bản sao" trong train
train_s = splits["train"]["sentence"].tolist()
test_s = test["sentence"].tolist()
print("trùng khớp từng ký tự trong train:", len(train_s) - len(set(train_s)), "câu dư")
for thr in (0.99, 0.95, 0.90):
    t0 = time.perf_counter()
    pairs = near_duplicates(emb["train"], thr)
    same = np.mean([y["train"][i] == y["train"][j] for i, j, _ in pairs]) if pairs else float("nan")
    print(f"cặp cosine >= {thr}: {len(pairs):6d} ({time.perf_counter() - t0:.1f} s), cùng nhãn cảm xúc: {same:.3f}")
pairs = near_duplicates(emb["train"], 0.95)
for i, j, s in [p for p in pairs if train_s[p[0]] != train_s[p[1]]][:3]:
    print(f"  {s:.3f} | {train_s[i]!r} | {train_s[j]!r}")

j, sim = nearest_in(emb["train"], emb["test"])
for thr in (0.99, 0.95, 0.90):
    leak = sim >= thr
    agree = (y["train"][j[leak]] == y["test"][leak]).mean()
    print(f"câu test có câu train cosine >= {thr}: {leak.sum():4d}/{len(test)}; nhãn 1-NN khớp ở nhóm này: {agree:.3f}")
leak = sim >= 0.95
print("macro-F1 LogReg trên câu test có bản sao trong train:", round(f1_score(y["test"][leak], pred_lr[leak], average="macro"), 3),
      "| trên phần còn lại:", round(f1_score(y["test"][~leak], pred_lr[~leak], average="macro"), 3))

## 3. Phát hiện bất thường bằng khoảng cách kNN

- `knn_anomaly_scores(ref, x, k, batch=2048) -> np.ndarray`: điểm bất thường của mỗi dòng của `x` là
  $$1 - \frac{1}{k}\sum_{j \in \mathrm{top}k} \cos(x, r_j)$$, tức cosine distance trung bình tới `k` láng giềng gần nhất
  trong `ref`. Điểm càng cao càng lạ.
- `auroc(scores, labels) -> float`: diện tích dưới đường ROC khi `labels == 1` là bất thường, tính bằng thứ hạng
  (Mann-Whitney): xác suất một mẫu bất thường ngẫu nhiên có điểm cao hơn một mẫu bình thường ngẫu nhiên, bằng điểm tính
  một nửa. Không dùng scikit-learn trong hàm này.

In [ ]:
def knn_anomaly_scores(ref: np.ndarray, x: np.ndarray, k: int, batch: int = 2048) -> np.ndarray:
    """1 - cosine trung bình tới k láng giềng gần nhất trong ref."""
    ref = np.asarray(ref, dtype=np.float32)
    out = []
    for s in range(0, len(x), batch):
        sims = np.asarray(x[s:s + batch], dtype=np.float32) @ ref.T
        top = -np.partition(-sims, k - 1, axis=1)[:, :k]
        out.append(1.0 - top.mean(axis=1))
    return np.concatenate(out)


def auroc(scores: np.ndarray, labels: np.ndarray) -> float:
    """AUROC theo thứ hạng (Mann-Whitney U), bằng điểm tính một nửa."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels).astype(bool)
    pos, neg = scores[labels], scores[~labels]
    greater = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    return float((greater + 0.5 * ties) / (len(pos) * len(neg)))

In [ ]:
# explore: hai bài toán bất thường: câu hỏi pháp luật lẫn vào phản hồi sinh viên, và chủ đề lạ
zalo_q = [json.loads(line)["text"] for line in (ZALO / "queries.jsonl").open(encoding="utf-8")][:300]
zalo_emb = llmkit.embed(zalo_q)
zalo_emb /= np.linalg.norm(zalo_emb, axis=1, keepdims=True)
normal = emb["test"][:300]
scores = knn_anomaly_scores(emb["train"], np.concatenate([normal, zalo_emb]), k=10)
labels = np.r_[np.zeros(300), np.ones(300)]
print("câu hỏi Zalo (ngoài miền) lẫn vào 300 câu test UIT-VSFC: AUROC =", round(auroc(scores, labels), 3))

topic = {s: splits[s]["topic"].to_numpy() for s in splits}
ref = emb["train"][topic["train"] == 0]                    # chỉ phản hồi về giảng viên làm "bình thường"
mask = np.isin(topic["test"], [0, 2])                       # test: giảng viên (0) và cơ sở vật chất (2)
scores = knn_anomaly_scores(ref, emb["test"][mask], k=10)
labels = (topic["test"][mask] == 2).astype(int)
print(f"tham chiếu = {len(ref)} câu train chủ đề giảng viên; test {mask.sum()} câu, trong đó {labels.sum()} câu cơ sở vật chất")
print("chủ đề cơ sở vật chất là bất thường: AUROC =", round(auroc(scores, labels), 3))
for k in (1, 10, 50):
    print(f"  k={k:3d}: AUROC = {auroc(knn_anomaly_scores(ref, emb['test'][mask], k=k), labels):.3f}")

## 4. Gợi ý: "điều luật liên quan"

`recommend(x, liked, k) -> list[int]`: hồ sơ người dùng là trung bình các vector `liked` (chỉ số dòng của `x`), chuẩn
hoá L2; trả `k` chỉ số có cosine với hồ sơ lớn nhất, **không** gồm các mục trong `liked`, giảm dần (bằng nhau thì chỉ
số nhỏ).

In [ ]:
def recommend(x: np.ndarray, liked: list[int], k: int) -> list[int]:
    """k mục gần hồ sơ (trung bình chuẩn hoá của liked) nhất, bỏ các mục đã thích."""
    profile = np.asarray(x[liked], dtype=np.float32).mean(axis=0)
    profile /= np.linalg.norm(profile)
    sims = np.asarray(x, dtype=np.float32) @ profile
    skip = set(int(i) for i in liked)
    return [int(i) for i in np.argsort(-sims, kind="stable") if int(i) not in skip][:k]

In [ ]:
# explore: với điều đúng của 300 câu hỏi ghim, gợi ý 5 điều liên quan; bao nhiêu phần cùng văn bản?
zids = json.loads((ZALO / "cache" / "bge-m3.ids.json").read_text(encoding="utf-8"))
zmat = np.load(ZALO / "cache" / "bge-m3.npy", mmap_mode="r")
row_of = {u: i for i, u in enumerate(zids)}
law = np.array([u.rsplit("+", 1)[0] for u in zids])
seeds = []
for line in (ZALO / "qrels" / "test.jsonl").open(encoding="utf-8"):
    r = json.loads(line)
    if r["query-id"] not in {s[0] for s in seeds}:
        seeds.append((r["query-id"], r["corpus-id"]))
seeds = seeds[:300]
zfull = np.asarray(zmat)
t0 = time.perf_counter()
same = [np.mean(law[recommend(zfull, [row_of[c]], 5)] == law[row_of[c]]) for _, c in seeds]
print(f"gợi ý 5 điều cho 300 điều gốc ({1000 * (time.perf_counter() - t0) / 300:.0f} ms/lần): cùng văn bản {np.mean(same):.3f}")
sizes = pd.Series(law).value_counts()
rand = np.mean([(sizes[law[row_of[c]]] - 1) / (len(zids) - 1) for _, c in seeds])
print(f"nếu gợi ý ngẫu nhiên: cùng văn bản {rand:.4f}")
i0 = row_of[seeds[0][1]]
print("ví dụ, điều gốc:", zids[i0], "->", [zids[i] for i in recommend(zfull, [i0], 5)])

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Embedding + một lớp tuyến tính đứng ở đâu so với TF-IDF, BiLSTM tự train và LLM 3B prompting? Lớp neutral (5% tập
  test) thì sao, và vì sao kNN và centroid tụt ở chính lớp đó?
- Bao nhiêu câu test có "bản sao" gần như y hệt trong train? Điểm trên nhóm đó so với phần còn lại cho thấy gì về một
  con số test đẹp?
- Vì sao phát hiện câu hỏi pháp luật lẫn vào phản hồi sinh viên dễ hơn nhiều so với phát hiện "chủ đề cơ sở vật chất"?

Sau đó làm phần trắc nghiệm của bài trong app.